In [36]:
# read the csv file
import pandas as pd

# read the CSV file
df = pd.read_csv('data_optim_add 1.csv')  #

# Filter conditions
filtered_df = df[(df['year'] == 2023) & 
                 (df['compensatedRelativePressure_mafiltered'].notna()) &
                (df['alpha'] != 0)] 

# Step 3: Create a new DataFrame (grouped by csemCur, ensuring non-empty parts for each csemCur)
new_df = filtered_df.groupby('csemCur', group_keys=False).apply(lambda x: x.dropna(subset=['compensatedRelativePressure_mafiltered']))

# load heritage data
donnees_patrimoniales = pd.read_csv('Donnees patrimoniales - volume.csv')

# convert csemCur to numeric for both DataFrames
new_df['csemCur'] = pd.to_numeric(new_df['csemCur'], errors='coerce')
donnees_patrimoniales['csemCur'] = pd.to_numeric(donnees_patrimoniales['csemCur'], errors='coerce')

# merge volume column based on csemCur (left join to keep all rows in new_df)
new_df = new_df.merge(
    donnees_patrimoniales[['csemCur', 'Volume']], 
    on='csemCur', 
    how='left'
)

# 新增：转换csemCur为int型（处理NaN）
new_df['csemCur'] = pd.to_numeric(new_df['csemCur'], errors='coerce').astype('Int64')  # Int64支持NaN

# save to a new CSV
new_df.to_csv('new_df_with_volume.csv', index=False)

# results
print("New DataFrame (new_df) created successfully!")
print(f"Shape: {new_df.shape}")
print(f"Unique csemCur count: {new_df['csemCur'].nunique()}")
print(new_df.head())



New DataFrame (new_df) created successfully!
Shape: (595, 13)
Unique csemCur count: 595
      csemCur  year  delta_pressure   cr  \
0  5000200795  2023             NaN  0.0   
1  5000252448  2023             NaN  0.0   
2  5000252449  2023             NaN  0.0   
3  5000252450  2023             NaN  0.0   
4  5000252524  2023             NaN  0.0   

   compensatedRelativePressure_mafiltered  COPER  MANX     alpha  \
0                                3.628186      0     0  0.004915   
1                                3.208579      0     0  0.181613   
2                                3.483369      0     0  0.022172   
3                                3.452095      0     0  0.018647   
4                                3.502114      0     0  0.032852   

   algo_critere_exploit_B_mafiltered  algo_critere_env_B_mafiltered  \
0                              False                          False   
1                              False                          False   
2                        

In [74]:
# import pandas as pd

# df = pd.read_excel("Donnees patrimoniales - edited.xlsm")
# df = df.rename(columns={'N° eqt': 'csemCur'})
# df.to_csv('Donnees patrimoniales - volume.csv', index=False)

# print("✅ Donnees patrimoniales - volume.csv")
# print("columns:", df.columns.tolist())


In [86]:
# Model 1: no constraint
from gurobipy import Model, GRB, quicksum

# -------------- Data Preparation --------------
# 0. 读取new_df_with_volume.csv
new_df = pd.read_csv('new_df_with_volume.csv', sep=',')

# ---------------- Parameters ----------------
N = 3                          # CSEM number

alpha_init = new_df['alpha'].head(N).tolist()
print("csemCur number:", new_df['csemCur'].head(N).tolist())
print("alpha_init:", alpha_init)
V = new_df['Volume'].head(N).tolist() 
print("Volume:", V) 
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]    # K[i] calculation


T_years = 27                    # planning years
Y = 365                        # time units per year
P_high, P_low = 3.5, 3.2       # pressure thresholds
Delta_P = P_high - P_low       # = 0.3
C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]  # [0.0952, 0.0992, ..., 0.1312]
C_leak_t = [c * 25200 for c in c_factors]  # [2399.04, 2499.84, ..., 3306.24]

mu_COPER = 1.6
mu_MANT  = 4.2

L_total = 100
M = 0.2                        

mdl = Model("yearly_gas_maintenance")

# -------------- Decision Variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, lb=0, name="alpha")  # alpha[i,t]: value for the year t

# -------------- Constraints --------------

# 1. Initial alpha
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i], name=f"alpha_init_{i}")

# 2. At most one maintenance per year/point
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(x_COPER[i,t] + x_MANT[i,t] <= 1, name=f"one_action_{i}_{t}")

# 3. alpha evolution
for i in range(N):
    for t in range(T_years):
        # No maintenance: alpha[t+1] = 1.01 * alpha[t]
        mdl.addConstr(alpha[i,t+1] >= 1.01 * alpha[i,t] - M * (x_COPER[i,t] + x_MANT[i,t]))
        mdl.addConstr(alpha[i,t+1] <= 1.01 * alpha[i,t] + M * (x_COPER[i,t] + x_MANT[i,t]))
        
        # COPER maintenance: alpha[t+1] = 1.01 * alpha[t] / mu_COPER
        alpha_coper = 1.01 * alpha[i,t] / mu_COPER
        mdl.addConstr(alpha[i,t+1] >= alpha_coper - M * (1 - x_COPER[i,t]))
        mdl.addConstr(alpha[i,t+1] <= alpha_coper + M * (1 - x_COPER[i,t]))
        
        # MANT maintenance: alpha[t+1] = 1.01 * alpha[t] / mu_MANT
        alpha_mant = 1.01 * alpha[i,t] / mu_MANT
        mdl.addConstr(alpha[i,t+1] >= alpha_mant - M * (1 - x_MANT[i,t]))
        mdl.addConstr(alpha[i,t+1] <= alpha_mant + M * (1 - x_MANT[i,t]))


# -------------- Objective: leak cost + maintenance cost --------------

# Annual leak cost: Leak_{i,t} = [Y * alpha[i,t] / Delta_P] * K[i] * C_leak
leak_cost = quicksum(
    (Y * alpha[i,t] / Delta_P) * K[i] * C_leak_t[t]  
    for i in range(N) for t in range(T_years)
)

# Maintenance cost
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)

mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)
mdl.Params.OutputFlag = 1  # Show solving process


# -------------- Solve & Output --------------
mdl.optimize()

# analyze infeasibility
# print(f"状态码: {mdl.status}")

# print(f"GRB常量: {GRB.INFEASIBLE}, {GRB.OPTIMAL}")

# if mdl.status == GRB.INFEASIBLE:
#     mdl.computeIIS()
#     print("\n=== 冲突约束 ===")
#     for c in mdl.getConstrs():
#         if c.IISConstr:
#             print(f"  {c.ConstrName}")

if mdl.status == GRB.OPTIMAL:
    print(f"Total cost: {mdl.objVal:.2f}")
    
    print("\n=== Maintenance Plan ===")
    for i in range(N):
        print(f"\nCSEM {i}:")
        for t in range(T_years):
            if x_COPER[i,t].X > 0.5:
                print(f"  Year {t+1}: COPER (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            elif x_MANT[i,t].X > 0.5:
                print(f"  Year {t+1}: MANT (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            else:
                print(f"  Year {t+1}: No maintenance (α={alpha[i,t+1].X:.6f})")
else:
    print("No optimal solution found")

csemCur number: [5000200795, 5000252448, 5000252449]
alpha_init: [0.0049148948437029, 0.1816128661238826, 0.022172033787185]
Volume: [7.58361265949119, 63.1967721624266, 63.0536851311155]
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 570 rows, 246 columns and 1785 nonzeros
Model fingerprint: 0xc72d1113
Variable types: 84 continuous, 162 integer (162 binary)
Coefficient statistics:
  Matrix range     [2e-01, 1e+00]
  Objective range  [2e+04, 2e+09]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e-03, 1e+00]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.
Presolve removed 385 rows and 165 columns
Presolve time: 0.09s
Presolved: 185 rows, 81 columns, 654 nonzeros
Variable types: 27 continuous, 54 integer (54 binary)
Found heuristic solution: objective 9.754309e+09

Root relaxation:

In [75]:
# Model 2: budget constraint
from gurobipy import Model, GRB, quicksum

# -------------- Data Preparation --------------
# 0. 读取new_df_with_volume.csv
new_df = pd.read_csv('new_df_with_volume.csv', sep=',')

# ---------------- Parameters ----------------
N = 3                          # CSEM number

lpha_init = new_df['alpha'].head(N).tolist()
print("csemCur number:", new_df['csemCur'].head(N).tolist())
print("alpha_init:", alpha_init)
V = new_df['Volume'].head(N).tolist() 
print("Volume:", V) 
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]    # K[i] calculation

T_years = 7                   # planning years
Y = 365                        # time units per year
P_high, P_low = 3.5, 3.2      # pressure thresholds
Delta_P = P_high - P_low       # = 0.3
C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]  # [0.0952, 0.0992, ..., 0.1312]
C_leak_t = [c * 25200 for c in c_factors]  # [2399.04, 2499.84, ..., 3306.24]

Total_budget = 11812500       # total budget
mu_COPER = 1.6
mu_MANT  = 4.2

L_total = 100
M = 0.2                        # Big-M (alpha range approximately 0~0.01)

mdl = Model("yearly_gas_maintenance")

# -------------- Decision Variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, lb=0, name="alpha")  # alpha[i,t]: value for the year t

# -------------- Constraints --------------

# 1. Initial alpha
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i], name=f"alpha_init_{i}")

# 2. At most one maintenance per year/point
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(x_COPER[i,t] + x_MANT[i,t] <= 1, name=f"one_action_{i}_{t}")

# 3. alpha evolution
for i in range(N):
    for t in range(T_years):
        # No maintenance: alpha[t+1] = 1.01 * alpha[t]
        mdl.addConstr(alpha[i,t+1] >= 1.01 * alpha[i,t] - M * (x_COPER[i,t] + x_MANT[i,t]))
        mdl.addConstr(alpha[i,t+1] <= 1.01 * alpha[i,t] + M * (x_COPER[i,t] + x_MANT[i,t]))
        
        # COPER maintenance: alpha[t+1] = 1.01 * alpha[t] / mu_COPER
        alpha_coper = 1.01 * alpha[i,t] / mu_COPER
        mdl.addConstr(alpha[i,t+1] >= alpha_coper - M * (1 - x_COPER[i,t]))
        mdl.addConstr(alpha[i,t+1] <= alpha_coper + M * (1 - x_COPER[i,t]))
        
        # MANT maintenance: alpha[t+1] = 1.01 * alpha[t] / mu_MANT
        alpha_mant = 1.01 * alpha[i,t] / mu_MANT
        mdl.addConstr(alpha[i,t+1] >= alpha_mant - M * (1 - x_MANT[i,t]))
        mdl.addConstr(alpha[i,t+1] <= alpha_mant + M * (1 - x_MANT[i,t]))


# -------------- Objective: leak cost + maintenance cost --------------

# Annual leak cost: Leak_{i,t} = [Y * alpha[i,t] / Delta_P] * K[i] * C_leak
leak_cost = quicksum(
    (Y * alpha[i,t] / Delta_P) * K[i] * C_leak_t[t]  
    for i in range(N) for t in range(T_years)
)

# Maintenance cost
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)

total_cost = action_cost + leak_cost

# 总成本预算约束
mdl.addConstr(action_cost <= Total_budget, name="TotalBudgetLimit")

mdl.setObjective(total_cost, GRB.MINIMIZE)
mdl.Params.OutputFlag = 1  # Show solving process

# -------------- Solve & Output --------------
mdl.optimize()

if mdl.status == GRB.OPTIMAL:
    print(f"Total cost: {mdl.objVal:.2f}")
    
    print("\n=== Maintenance Plan ===")
    for i in range(N):
        print(f"\nCSEM {i}:")
        for t in range(T_years):
            if x_COPER[i,t].X > 0.5:
                print(f"  Year {t+1}: COPER (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            elif x_MANT[i,t].X > 0.5:
                print(f"  Year {t+1}: MANT (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            else:
                print(f"  Year {t+1}: No maintenance (α={alpha[i,t+1].X:.6f})")
else:
    print("No optimal solution found")

csemCur number: [5000200795, 5000252448, 5000252449]
alpha_init: [0.0049148948437029, 0.1816128661238826, 0.022172033787185]
Volume: [7.58361265949119, 63.1967721624266, 63.0536851311155]
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 151 rows, 66 columns and 507 nonzeros
Model fingerprint: 0x1b714b8e
Variable types: 24 continuous, 42 integer (42 binary)
Coefficient statistics:
  Matrix range     [2e-01, 5e+04]
  Objective range  [2e+04, 1e+09]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e-03, 1e+07]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.
Found heuristic solution: objective 3.161364e+08
Presolve removed 106 rows and 45 columns
Presolve time: 0.01s
Presolved: 45 rows, 21 columns, 152 nonzeros
Found heuristic solution: objective 3.156846e+08
Variable types: 7 continuous

In [76]:
# Model 3: resources constraint

from gurobipy import Model, GRB, quicksum

# -------------- Data Preparation --------------
# 0. 读取new_df_with_volume.csv
new_df = pd.read_csv('new_df_with_volume.csv', sep=',')

# ---------------- Parameters ----------------
N = 3                          # CSEM number

alpha_init = new_df['alpha'].head(N).tolist()
print("csemCur number:", new_df['csemCur'].head(N).tolist())
print("alpha_init:", alpha_init)
V = new_df['Volume'].head(N).tolist() 
print("Volume:", V) 
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]    # K[i] calculation

T_years = 7                   # planning years
Y = 365                        # time units per year
P_high, P_low = 3.5, 3.2      # pressure thresholds
Delta_P = P_high - P_low       # = 0.3
C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]  # [0.0952, 0.0992, ..., 0.1312]
C_leak_t = [c * 25200 for c in c_factors]  # [2399.04, 2499.84, ..., 3306.24]

Total_budget = 11812500       # total budget
mu_COPER = 1.6
mu_MANT  = 4.2

L_total = 100
M = 0.2                        # Big-M (alpha range approximately 0~0.01)

mdl = Model("yearly_gas_maintenance")

# -------------- Decision Variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, lb=0, name="alpha")  # alpha[i,t]: value for the year t

# -------------- Constraints --------------

# 1. Initial alpha
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i], name=f"alpha_init_{i}")

# 2. At most one maintenance per year/point
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(x_COPER[i,t] + x_MANT[i,t] <= 1, name=f"one_action_{i}_{t}")

# 3. alpha evolution
for i in range(N):
    for t in range(T_years):
        # No maintenance: alpha[t+1] = 1.01 * alpha[t]
        mdl.addConstr(alpha[i,t+1] >= 1.01 * alpha[i,t] - M * (x_COPER[i,t] + x_MANT[i,t]))
        mdl.addConstr(alpha[i,t+1] <= 1.01 * alpha[i,t] + M * (x_COPER[i,t] + x_MANT[i,t]))
        
        # COPER maintenance: alpha[t+1] = 1.01 * alpha[t] / mu_COPER
        alpha_coper = 1.01 * alpha[i,t] / mu_COPER
        mdl.addConstr(alpha[i,t+1] >= alpha_coper - M * (1 - x_COPER[i,t]))
        mdl.addConstr(alpha[i,t+1] <= alpha_coper + M * (1 - x_COPER[i,t]))
        
        # MANT maintenance: alpha[t+1] = 1.01 * alpha[t] / mu_MANT
        alpha_mant = 1.01 * alpha[i,t] / mu_MANT
        mdl.addConstr(alpha[i,t+1] >= alpha_mant - M * (1 - x_MANT[i,t]))
        mdl.addConstr(alpha[i,t+1] <= alpha_mant + M * (1 - x_MANT[i,t]))

# 4. 总任务数限制
mdl.addConstr(
    quicksum(x_COPER[i,t] + x_MANT[i,t] for i in range(N) for t in range(T_years)) <= L_total,
    name="TotalTasksLimit"
)

# -------------- Objective: leak cost + maintenance cost --------------

# Annual leak cost: Leak_{i,t} = [Y * alpha[i,t] / Delta_P] * K[i] * C_leak
leak_cost = quicksum(
    (Y * alpha[i,t] / Delta_P) * K[i] * C_leak_t[t]  
    for i in range(N) for t in range(T_years)
)

# Maintenance cost
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)

total_cost = action_cost + leak_cost

mdl.setObjective(total_cost, GRB.MINIMIZE)
mdl.Params.OutputFlag = 1  # Show solving process

# -------------- Solve & Output --------------
mdl.optimize()

if mdl.status == GRB.OPTIMAL:
    print(f"Total cost: {mdl.objVal:.2f}")
    
    print("\n=== Maintenance Plan ===")
    for i in range(N):
        print(f"\nCSEM {i}:")
        for t in range(T_years):
            if x_COPER[i,t].X > 0.5:
                print(f"  Year {t+1}: COPER (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            elif x_MANT[i,t].X > 0.5:
                print(f"  Year {t+1}: MANT (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            else:
                print(f"  Year {t+1}: No maintenance (α={alpha[i,t+1].X:.6f})")
else:
    print("No optimal solution found")

csemCur number: [5000200795, 5000252448, 5000252449]
alpha_init: [0.0049148948437029, 0.1816128661238826, 0.022172033787185]
Volume: [7.58361265949119, 63.1967721624266, 63.0536851311155]
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 151 rows, 66 columns and 507 nonzeros
Model fingerprint: 0x97606ee5
Variable types: 24 continuous, 42 integer (42 binary)
Coefficient statistics:
  Matrix range     [2e-01, 1e+00]
  Objective range  [2e+04, 1e+09]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e-03, 1e+02]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.
Found heuristic solution: objective 3.161364e+08
Presolve removed 106 rows and 45 columns
Presolve time: 0.01s
Presolved: 45 rows, 21 columns, 152 nonzeros
Found heuristic solution: objective 3.156846e+08
Variable types: 7 continuous

In [77]:
# Model 4: budget and resources constraints
from gurobipy import Model, GRB, quicksum

# -------------- Data Preparation --------------
# 0. 读取new_df_with_volume.csv
new_df = pd.read_csv('new_df_with_volume.csv', sep=',')

# ---------------- Parameters ----------------
N = 3                          # CSEM number

alpha_init = new_df['alpha'].head(N).tolist()
print("csemCur number:", new_df['csemCur'].head(N).tolist())
print("alpha_init:", alpha_init)
V = new_df['Volume'].head(N).tolist() 
print("Volume:", V) 
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]    # K[i] calculation

T_years = 7                   # planning years
Y = 365                        # time units per year
P_high, P_low = 3.5, 3.2      # pressure thresholds
Delta_P = P_high - P_low       # = 0.3
C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]  # [0.0952, 0.0992, ..., 0.1312]
C_leak_t = [c * 25200 for c in c_factors]  # [2399.04, 2499.84, ..., 3306.24]

Total_budget = 2_000_000       # total budget
mu_COPER = 1.6
mu_MANT  = 4.2

L_total = 100
M = 0.2                        # Big-M (alpha range approximately 0~0.01)

mdl = Model("yearly_gas_maintenance")

# -------------- Decision Variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, lb=0, name="alpha")  # alpha[i,t]: value for the year t

# -------------- Constraints --------------

# 1. Initial alpha
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i], name=f"alpha_init_{i}")

# 2. At most one maintenance per year/point
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(x_COPER[i,t] + x_MANT[i,t] <= 1, name=f"one_action_{i}_{t}")

# 3. alpha evolution
for i in range(N):
    for t in range(T_years):
        # No maintenance: alpha[t+1] = 1.01 * alpha[t]
        mdl.addConstr(alpha[i,t+1] >= 1.01 * alpha[i,t] - M * (x_COPER[i,t] + x_MANT[i,t]))
        mdl.addConstr(alpha[i,t+1] <= 1.01 * alpha[i,t] + M * (x_COPER[i,t] + x_MANT[i,t]))
        
        # COPER maintenance: alpha[t+1] = 1.01 * alpha[t] / mu_COPER
        alpha_coper = 1.01 * alpha[i,t] / mu_COPER
        mdl.addConstr(alpha[i,t+1] >= alpha_coper - M * (1 - x_COPER[i,t]))
        mdl.addConstr(alpha[i,t+1] <= alpha_coper + M * (1 - x_COPER[i,t]))
        
        # MANT maintenance: alpha[t+1] = 1.01 * alpha[t] / mu_MANT
        alpha_mant = 1.01 * alpha[i,t] / mu_MANT
        mdl.addConstr(alpha[i,t+1] >= alpha_mant - M * (1 - x_MANT[i,t]))
        mdl.addConstr(alpha[i,t+1] <= alpha_mant + M * (1 - x_MANT[i,t]))

# 4. 总任务数限制
mdl.addConstr(
    quicksum(x_COPER[i,t] + x_MANT[i,t] for i in range(N) for t in range(T_years)) <= L_total,
    name="TotalTasksLimit"
)

# -------------- Objective: leak cost + maintenance cost --------------

# Annual leak cost: Leak_{i,t} = [Y * alpha[i,t] / Delta_P] * K[i] * C_leak
leak_cost = quicksum(
    (Y * alpha[i,t] / Delta_P) * K[i] * C_leak_t[t]  
    for i in range(N) for t in range(T_years)
)


# Maintenance cost
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)

total_cost = action_cost + leak_cost

# 总成本预算约束
mdl.addConstr(action_cost <= Total_budget, name="TotalBudgetLimit")

mdl.setObjective(total_cost, GRB.MINIMIZE)
mdl.Params.OutputFlag = 1  # Show solving process

# -------------- Solve & Output --------------
mdl.optimize()

if mdl.status == GRB.OPTIMAL:
    print(f"Total cost: {mdl.objVal:.2f}")
    
    print("\n=== Maintenance Plan ===")
    for i in range(N):
        print(f"\nCSEM {i}:")
        for t in range(T_years):
            if x_COPER[i,t].X > 0.5:
                print(f"  Year {t+1}: COPER (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            elif x_MANT[i,t].X > 0.5:
                print(f"  Year {t+1}: MANT (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            else:
                print(f"  Year {t+1}: No maintenance (α={alpha[i,t+1].X:.6f})")
else:
    print("No optimal solution found")

csemCur number: [5000200795, 5000252448, 5000252449]
alpha_init: [0.0049148948437029, 0.1816128661238826, 0.022172033787185]
Volume: [7.58361265949119, 63.1967721624266, 63.0536851311155]
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 152 rows, 66 columns and 549 nonzeros
Model fingerprint: 0x33c174ed
Variable types: 24 continuous, 42 integer (42 binary)
Coefficient statistics:
  Matrix range     [2e-01, 5e+04]
  Objective range  [2e+04, 1e+09]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e-03, 2e+06]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.
Found heuristic solution: objective 3.161364e+08
Presolve removed 107 rows and 45 columns
Presolve time: 0.01s
Presolved: 45 rows, 21 columns, 152 nonzeros
Found heuristic solution: objective 3.156846e+08
Variable types: 7 continuous